In [2]:
"""
Data Checking Code - Dataset Overview and Validation
Date: Nov 17 2025
Updated: June 29 2026

Input Files:
compilation_Venuti_2024_Serna_2021.csv

Purpose:
- Load and validate the stellar accretion dataset
- Count data points by spectral class (first letter of SpT)
- Count data points by mass bins (≤2.0 M☉ and >2.0 M☉)
- Count disk presence categories
- Count accretion status categories

This provides a quick overview of the dataset before running the main analysis.
"""

import pandas as pd
import os


def load_dataset(file_path):
    """
    Load the dataset from a CSV file with error handling.
    
    Parameters:
        file_path (str): Path to the CSV file
        
    Returns:
        pd.DataFrame: Loaded dataframe, or None if loading fails
    """
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' not found.")
        print(f"Current directory: {os.getcwd()}")
        return None
    
    try:
        df = pd.read_csv(file_path)
        return df
    except Exception as e:
        print(f"Error loading file: {e}")
        return None


def validate_columns(df, required_cols):
    """
    Check for required columns and warn if any are missing.
    
    Parameters:
        df (pd.DataFrame): Input dataframe
        required_cols (list): List of required column names
        
    Returns:
        list: List of missing columns
    """
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"Warning: Missing columns: {missing_cols}")
    return missing_cols


def sort_dataframe(df, sort_cols):
    """
    Sort dataframe by specified columns.
    
    Parameters:
        df (pd.DataFrame): Input dataframe
        sort_cols (list): List of column names to sort by
        
    Returns:
        pd.DataFrame: Sorted dataframe
    """
    existing_cols = [c for c in sort_cols if c in df.columns]
    if existing_cols:
        return df.sort_values(existing_cols).reset_index(drop=True)
    return df


def count_spectral_classes(df):
    """
    Count data points by spectral class (first letter of SpT).
    
    Parameters:
        df (pd.DataFrame): Input dataframe with 'SpT' column
        
    Returns:
        pandas.Series: Counts per spectral class
    """
    valid_classes = ['O', 'B', 'A', 'F', 'G', 'K', 'M']
    
    df['SpT_class'] = df['SpT'].astype(str).str.strip().str[0]
    df['SpT_class'] = df['SpT_class'].where(
        df['SpT_class'].isin(valid_classes), 'Other'
    )
    df.loc[df['SpT'].isna(), 'SpT_class'] = 'NaN'
    
    return df['SpT_class'].value_counts()


def count_mass_bins(df):
    """
    Count data points by mass bins and compute mass statistics.
    
    Parameters:
        df (pd.DataFrame): Input dataframe with 'Mstar' column
        
    Returns:
        tuple: (valid_mass, below_2, above_2) DataFrames for each bin
    """
    valid_mass = df['Mstar'].dropna()
    
    below_2 = valid_mass[valid_mass <= 2.0]
    above_2 = valid_mass[valid_mass > 2.0]
    
    return valid_mass, below_2, above_2


def count_categories(df, column, mapping, label):
    """
    Count occurrences of categories in a column.
    
    Parameters:
        df (pd.DataFrame): Input dataframe
        column (str): Column name to count
        mapping (dict): Mapping of values to labels
        label (str): Label for the category being counted
        
    Returns:
        dict: Counts per category
    """
    counts = df[column].value_counts(dropna=False)
    results = {}
    
    for key, label_text in mapping.items():
        results[label_text] = counts.get(key, 0)
    
    # Check for unexpected values
    unexpected = [x for x in counts.index if x not in mapping and not pd.isna(x)]
    if unexpected:
        print(f"\nWarning: Unexpected {label} values: {unexpected}")
        for val in unexpected:
            print(f"  {val}: {counts.get(val, 0)}")
    
    # Handle NaN separately
    if pd.isna(counts.index).any():
        print(f"NaN: {counts.get(pd.NA, 0)}")
    
    return results


def print_completeness_summary(df):
    """
    Print data completeness summary for main analysis columns.
    
    Parameters:
        df (pd.DataFrame): Input dataframe
    """
    main_analysis_cols = ['logAge', 'Mstar', 'logMacc']
    missing_main = [col for col in main_analysis_cols if col not in df.columns]
    
    if missing_main:
        print(f"Warning: Missing required columns: {missing_main}")
        return
    
    complete_data = df.dropna(subset=main_analysis_cols)
    print(f"Complete rows for main analysis: {len(complete_data)}/{len(df)} ({len(complete_data)/len(df)*100:.1f}%)")
    
    print("\nMissing data per column:")
    for col in df.columns:
        missing = df[col].isna().sum()
        if missing > 0:
            print(f"  {col}: {missing} ({missing/len(df)*100:.1f}%)")
    
    print("\nColumns with complete data:")
    for col in df.columns:
        if df[col].isna().sum() == 0:
            print(f"  {col}")


def main():
    """Load and analyze the dataset."""
    
    # Load dataset
    file_path = 'compilation_Venuti_2024_Serna_2021.csv'
    df = load_dataset(file_path)
    if df is None:
        return
    
    # Validate columns
    validate_columns(df, ['Mstar', 'Disk', 'Acc'])
    
    # Sort by mass then age
    df = sort_dataframe(df, ['Mstar', 'logAge'])
    
    print("="*60)
    print("DATASET OVERVIEW AFTER INITIAL SORTING")
    print("="*60)
    print(f"Total datapoints: {len(df)}")
    
    # Spectral class counts
    print("\n" + "-"*60)
    print("DATAPOINTS PER SPECTRAL CLASS (FIRST LETTER OF SpT)")
    print("-"*60)
    
    if 'SpT' in df.columns:
        spt_class_counts = count_spectral_classes(df)
        print("OBAFGKM Classes:")
        for cls in ['O', 'B', 'A', 'F', 'G', 'K', 'M']:
            print(f"  {cls}: {spt_class_counts.get(cls, 0)}")
        print(f"  NaN: {spt_class_counts.get('NaN', 0)}")
        print(f"  Other: {spt_class_counts.get('Other', 0)}")
    else:
        print("Warning: 'SpT' column not found")
    
    # Mass bin counts
    print("\n" + "-"*60)
    print("DATAPOINTS PER MASS BIN")
    print("-"*60)
    
    if 'Mstar' in df.columns:
        valid_mass, below_2, above_2 = count_mass_bins(df)
        print(f"Valid Mstar entries: {len(valid_mass)} (NaN: {df['Mstar'].isna().sum()})")
        print(f"Mstar ≤ 2.0 M☉ : {len(below_2)}")
        print(f"Mstar > 2.0 M☉ : {len(above_2)}")
        
        if len(valid_mass) > 0:
            print(f"\nMass Statistics:")
            print(f"  Min Mstar: {valid_mass.min():.3f} M☉")
            print(f"  Max Mstar: {valid_mass.max():.3f} M☉")
            print(f"  Mean Mstar: {valid_mass.mean():.3f} M☉")
            print(f"  Median Mstar: {valid_mass.median():.3f} M☉")
    else:
        print("Warning: 'Mstar' column not found")
    
    # Disk presence counts
    print("\n" + "-"*60)
    print("DISK PRESENCE COUNTS")
    print("-"*60)
    
    if 'Disk' in df.columns:
        disk_map = {'y': 'Disk = Yes', 'n': 'Disk = No', 'y*': 'Disk = Maybe'}
        disk_counts = count_categories(df, 'Disk', disk_map, 'disk')
        
        for label, count in disk_counts.items():
            print(f"{label}: {count}")
    else:
        print("Warning: 'Disk' column not found")
    
    # Accretion status counts
    print("\n" + "-"*60)
    print("ACCRETION STATUS COUNTS")
    print("-"*60)
    
    if 'Acc' in df.columns:
        acc_map = {0: 'Accretion = No', 1: 'Accretion = Yes', 2: 'Accretion = Potential'}
        acc_counts = count_categories(df, 'Acc', acc_map, 'accretion')
        
        for label, count in acc_counts.items():
            print(f"{label}: {count}")
    else:
        print("Warning: 'Acc' column not found")
    
    # Data completeness summary
    print("\n" + "-"*60)
    print("DATA COMPLETENESS SUMMARY")
    print("-"*60)
    
    print_completeness_summary(df)
    
    print("\nAnalysis complete.")


if __name__ == "__main__":
    main()

DATASET OVERVIEW AFTER INITIAL SORTING
Total datapoints: 1217

------------------------------------------------------------
DATAPOINTS PER SPECTRAL CLASS (FIRST LETTER OF SpT)
------------------------------------------------------------
OBAFGKM Classes:
  O: 6
  B: 45
  A: 77
  F: 112
  G: 137
  K: 487
  M: 101
  NaN: 251
  Other: 1

------------------------------------------------------------
DATAPOINTS PER MASS BIN
------------------------------------------------------------
Valid Mstar entries: 1128 (NaN: 89)
Mstar ≤ 2.0 M☉ : 944
Mstar > 2.0 M☉ : 184

Mass Statistics:
  Min Mstar: 0.149 M☉
  Max Mstar: 14.267 M☉
  Mean Mstar: 1.247 M☉
  Median Mstar: 0.901 M☉

------------------------------------------------------------
DISK PRESENCE COUNTS
------------------------------------------------------------

  n*: 155
  ?: 47
NaN: 0
Disk = Yes: 147
Disk = No: 413
Disk = Maybe: 82

------------------------------------------------------------
ACCRETION STATUS COUNTS
-------------------------